In [1]:
import random
import torch
from transformers import pipeline
from datasets import load_dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

if torch.backends.mps.is_available():
    device = "mps"
    pipeline_device = torch.device("mps")
else:
    device = "cpu"
    pipeline_device = -1

print(f"Using device: {device}")
print(f"Seed: {SEED}")

Using device: mps
Seed: 42


In [2]:
model_name = "cross-encoder/ms-marco-MiniLM-L6-v2"

clf = pipeline(
    task="text-classification",
    model=model_name,
    tokenizer=model_name,
    device=pipeline_device,
    truncation=True,
    max_length=128,
    return_all_scores=False
)

print(f"Loaded pipeline model: {model_name}")
print(f"Model labels: {clf.model.config.id2label}")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Loaded pipeline model: cross-encoder/ms-marco-MiniLM-L6-v2
Model labels: {0: 'LABEL_0'}


In [3]:
dataset = load_dataset("glue", "mrpc", split="validation")
print("Dataset split: glue/mrpc validation")
print(f"Number of examples in full validation split: {len(dataset)}")
print("Example row:")
print(dataset[0])

label0_indices = [i for i, row in enumerate(dataset) if row["label"] == 0]
label1_indices = [i for i, row in enumerate(dataset) if row["label"] == 1]

per_class_n = min(len(label0_indices), len(label1_indices), 100)
sampled_indices = random.sample(label0_indices, per_class_n) + random.sample(label1_indices, per_class_n)
random.shuffle(sampled_indices)

subset = dataset.select(sampled_indices)
subset_labels = subset["label"]
subset_label0 = sum(1 for x in subset_labels if x == 0)
subset_label1 = sum(1 for x in subset_labels if x == 1)

print(f"Balanced subset size: {len(subset)}")
print(f"Class balance -> label 0: {subset_label0}, label 1: {subset_label1}")

Dataset split: glue/mrpc validation
Number of examples in full validation split: 408
Example row:
{'sentence1': "He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .", 'sentence2': '" The foodservice pie business does not fit our long-term growth strategy .', 'label': 1, 'idx': 9}
Balanced subset size: 200
Class balance -> label 0: 100, label 1: 100


In [4]:
pairs = [{"text": row["sentence1"], "text_pair": row["sentence2"]} for row in subset]
labels = subset["label"]

print("Prepared pairwise inputs for pipeline inference.")
print(pairs[0])

Prepared pairwise inputs for pipeline inference.
{'text': 'In fiction : Edward P. Jones ( " The Known World " ) and Scott Spencer ( " A Ship Made of Paper " ) .', 'text_pair': 'The fifth nominee for fiction is Scott Spencer , for A Ship Made of Paper .'}


In [5]:
batch_size = 32
raw_outputs = clf(pairs, batch_size=batch_size)

def normalize_label_name(x):
    return str(x).upper()

label_to_id = {normalize_label_name(k): v for k, v in clf.model.config.label2id.items()}

def map_pipeline_label_to_binary(label_str):
    key = normalize_label_name(label_str)
    if key in label_to_id:
        model_id = label_to_id[key]
        if model_id == 1:
            return 1
        if model_id == 0:
            return 0
    if key in {"LABEL_1", "1", "POSITIVE", "RELEVANT", "TRUE", "PARAPHRASE"}:
        return 1
    if key in {"LABEL_0", "0", "NEGATIVE", "NOT_RELEVANT", "FALSE", "NOT_PARAPHRASE"}:
        return 0
    raise ValueError(f"Unsupported pipeline label: {label_str}")

predictions = [map_pipeline_label_to_binary(o["label"]) for o in raw_outputs]
confidences = [float(o["score"]) for o in raw_outputs]

print(f"Completed pipeline inference for {len(predictions)} examples.")
print("First 5 raw outputs:")
print(raw_outputs[:5])

Completed pipeline inference for 200 examples.
First 5 raw outputs:
[{'label': 'LABEL_0', 'score': 0.9962444305419922}, {'label': 'LABEL_0', 'score': 0.9999649524688721}, {'label': 'LABEL_0', 'score': 0.9979450106620789}, {'label': 'LABEL_0', 'score': 0.9972559809684753}, {'label': 'LABEL_0', 'score': 0.9154876470565796}]


In [6]:
accuracy = accuracy_score(labels, predictions)
precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average="binary", zero_division=0)
cm = confusion_matrix(labels, predictions)
avg_confidence = sum(confidences) / len(confidences)

pred0_conf = [c for p, c in zip(predictions, confidences) if p == 0]
pred1_conf = [c for p, c in zip(predictions, confidences) if p == 1]

def summarize_conf(xs):
    if not xs:
        return {"count": 0, "avg": None, "min": None, "max": None}
    return {
        "count": len(xs),
        "avg": sum(xs) / len(xs),
        "min": min(xs),
        "max": max(xs),
    }

pred0_summary = summarize_conf(pred0_conf)
pred1_summary = summarize_conf(pred1_conf)

print("Evaluation metrics on balanced subset:")
print(f"Accuracy              : {accuracy:.4f}")
print(f"Precision             : {precision:.4f}")
print(f"Recall                : {recall:.4f}")
print(f"F1                    : {f1:.4f}")
print(f"Average confidence    : {avg_confidence:.4f}")
print("Confusion matrix:")
print(cm)
print("Confidence summary by predicted class:")
print(f"Predicted class 0: {pred0_summary}")
print(f"Predicted class 1: {pred1_summary}")

Evaluation metrics on balanced subset:
Accuracy              : 0.5000
Precision             : 0.0000
Recall                : 0.0000
F1                    : 0.0000
Average confidence    : 0.9592
Confusion matrix:
[[100   0]
 [100   0]]
Confidence summary by predicted class:
Predicted class 0: {'count': 200, 'avg': 0.9592100764811039, 'min': 0.08097928762435913, 'max': 0.9999650716781616}
Predicted class 1: {'count': 0, 'avg': None, 'min': None, 'max': None}


In [7]:
label_map = {0: "not_paraphrase", 1: "paraphrase"}
errors = []

for i, (true_label, pred_label, conf) in enumerate(zip(labels, predictions, confidences)):
    if true_label != pred_label:
        row = subset[i]
        errors.append({
            "index_in_subset": i,
            "sentence1": row["sentence1"],
            "sentence2": row["sentence2"],
            "true_label": true_label,
            "pred_label": pred_label,
            "confidence": conf,
        })

errors = sorted(errors, key=lambda x: x["confidence"], reverse=True)
num_examples_to_show = min(5, len(errors))

print(f"Total errors: {len(errors)}")
print(f"Showing {num_examples_to_show} highest-confidence errors")

for example in errors[:num_examples_to_show]:
    print(f"Index in subset: {example['index_in_subset']}")
    print(f"sentence1: {example['sentence1']}")
    print(f"sentence2: {example['sentence2']}")
    print(f"true label: {example['true_label']} ({label_map[example['true_label']]})")
    print(f"pred label: {example['pred_label']} ({label_map[example['pred_label']]})")
    print(f"confidence: {example['confidence']:.4f}")
    print("-" * 80)

Total errors: 100
Showing 5 highest-confidence errors
Index in subset: 139
sentence1: Blair 's Foreign Secretary Jack Straw was to take his place on Monday to give a statement to parliament on the European Union .
sentence2: Blair 's office said his Foreign Secretary Jack Straw would take his place on Monday to give a statement to parliament on the EU meeting the prime minister attended last week .
true label: 1 (paraphrase)
pred label: 0 (not_paraphrase)
confidence: 1.0000
--------------------------------------------------------------------------------
Index in subset: 1
sentence1: " I think it 's going to be a close vote , but I think the grant proposal is going to win , " McConnell said .
sentence2: " I think it 's going to be a close vote , but I think the grant proposal 's going to win , " said Sen. Mitch McConnell , assistant majority leader .
true label: 1 (paraphrase)
pred label: 0 (not_paraphrase)
confidence: 1.0000
-------------------------------------------------------------

In [8]:
print("RESULT SUMMARY")
print(f"model={model_name}")
print("dataset_split=glue/mrpc validation")
print("dataset_variant=single_balanced_subset")
print(f"subset_size={len(subset)}")
print(f"subset_label_0={subset_label0}")
print(f"subset_label_1={subset_label1}")
print("inference_method=transformers.pipeline_text_classification_pairwise")
print(f"device={device}")
print(f"batch_size={batch_size}")
print(f"accuracy={accuracy:.4f}")
print(f"precision={precision:.4f}")
print(f"recall={recall:.4f}")
print(f"f1={f1:.4f}")
print(f"average_confidence={avg_confidence:.4f}")
print(f"predicted_class_0_confidence_summary={pred0_summary}")
print(f"predicted_class_1_confidence_summary={pred1_summary}")
print(f"num_errors={len(errors)}")

RESULT SUMMARY
model=cross-encoder/ms-marco-MiniLM-L6-v2
dataset_split=glue/mrpc validation
dataset_variant=single_balanced_subset
subset_size=200
subset_label_0=100
subset_label_1=100
inference_method=transformers.pipeline_text_classification_pairwise
device=mps
batch_size=32
accuracy=0.5000
precision=0.0000
recall=0.0000
f1=0.0000
average_confidence=0.9592
predicted_class_0_confidence_summary={'count': 200, 'avg': 0.9592100764811039, 'min': 0.08097928762435913, 'max': 0.9999650716781616}
predicted_class_1_confidence_summary={'count': 0, 'avg': None, 'min': None, 'max': None}
num_errors=100
